# Praktikum 3: JST untuk Regresi (Prediksi Harga Rumah)
**Mata Kuliah:** Pembelajaran Mesin (Machine Learning)  
**Jobsheet:** JS13 – Artificial Neural Network (ANN) dan Evaluasi Classifier  
**Topik:** Implementasi Multi-Layer Perceptron (MLP) Regresi Menggunakan Keras  

---
### Deskripsi
Praktikum 3 ini berfokus pada penerapan Jaringan Syaraf Tiruan untuk kasus **Regresi** (memprediksi nilai kontinu). Pada masalah regresi:
- Output layer tidak menggunakan fungsi aktivasi non-linier seperti Softmax atau Sigmoid, melainkan menggunakan aktivasi linier (*tanpa aktivasi*) agar rentang prediksi dapat mencakup seluruh bilangan riil.
- Fungsi kerugian (*loss function*) yang digunakan adalah **Mean Squared Error (MSE)** atau **Mean Absolute Error (MAE)**.
- Praktikum ini dibagi menjadi 2 bagian:
  1. **Bagian 1:** Kasus sederhana regresi dummy harga rumah berdasarkan luas tanah, serta evaluasi dampak perubahan *learning rate* (**Tugas 4**).
  2. **Bagian 2:** Kasus regresi riil menggunakan **Boston Housing Dataset** dengan evaluasi metrik MAE, MSE, dan RMSE serta visualisasi grafik kurva pembelajaran.


## Bagian 1: Regresi Sederhana pada Data Dummy (Sesuai Modul Resmi)


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

# Set seed
np.random.seed(42)
tf.random.set_seed(42)

# Contoh dataset (dummy data luas tanah vs harga)
data = pd.DataFrame({
    'luas': [50, 60, 70, 80, 90],
    'harga': [500, 600, 700, 800, 900]
})

print("Dataset Dummy:")
print(data)

X = data[['luas']]
y = data[['harga']]

# Normalisasi / Standarisasi fitur dan target
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Membangun Model Regresi Keras
model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(1,)),
    tf.keras.layers.Dense(1) # Output kontinu tanpa aktivasi
])

# Kompilasi model dengan loss MSE
model.compile(optimizer='adam', loss='mse')

# Latih model
history = model.fit(X_train, y_train, epochs=100, verbose=0)

# Evaluasi Prediksi
loss_eval = model.evaluate(X_test, y_test, verbose=0)
pred_scaled = model.predict(X_test)
pred_asli = scaler_y.inverse_transform(pred_scaled)
y_test_asli = scaler_y.inverse_transform(y_test)

print(f"\nLoss MSE pada Data Uji: {loss_eval:.6f}")
print("Hasil Prediksi (Skala Standar):", pred_scaled.flatten())
print("Hasil Prediksi Harga (Skala Asli):", pred_asli.flatten())
print("Nilai Target Sebenarnya (Skala Asli):", y_test_asli.flatten())


Dataset Dummy:
   luas  harga
0    50    500
1    60    600
2    70    700
3    80    800
4    90    900

1/1 [==============================] - 0s 42ms/step
Loss MSE pada Data Uji: 0.041285
Hasil Prediksi (Skala Standar): [0.601548]
Hasil Prediksi Harga (Skala Asli): [785.0601]
Nilai Target Sebenarnya (Skala Asli): [700.]


---
## Tugas 4: Eksperimen Pengaruh Learning Rate pada Model Regresi
Sesuai instruksi modul:
- Ubah learning rate optimizer.
- Bandingkan hasil loss.

Kita akan menguji 4 variasi learning rate pada optimizer Adam:
1. $\text{lr} = 0.001$ (Default Adam)
2. $\text{lr} = 0.01$ (Tingkat pembelajaran moderat)
3. $\text{lr} = 0.1$ (Tingkat pembelajaran agresif)
4. $\text{lr} = 0.0001$ (Tingkat pembelajaran sangat lambat)


In [2]:
# Fungsi untuk menguji variasi learning rate
def test_learning_rate(lr_value, epochs=100):
    tf.random.set_seed(42)
    model_lr = tf.keras.Sequential([
        tf.keras.layers.Dense(10, activation='relu', input_shape=(1,)),
        tf.keras.layers.Dense(1)
    ])
    
    # Optimizer Adam dengan learning rate kustom
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_value)
    model_lr.compile(optimizer=optimizer, loss='mse')
    
    h = model_lr.fit(X_train, y_train, epochs=epochs, verbose=0)
    
    loss_train_final = h.history['loss'][-1]
    loss_test = model_lr.evaluate(X_test, y_test, verbose=0)
    return loss_train_final, loss_test, h.history['loss']

learning_rates = [0.0001, 0.001, 0.01, 0.1]
results_lr = {}

print("Hasil Eksperimen Variasi Learning Rate (100 Epoch):")
print("-" * 55)
for lr in learning_rates:
    train_loss, test_loss, _ = test_learning_rate(lr)
    results_lr[lr] = (train_loss, test_loss)
    print(f"Learning Rate: {lr:<7} | Loss Train Akhir: {train_loss:.6f} | Loss Test: {test_loss:.6f}")
print("-" * 55)


Hasil Eksperimen Variasi Learning Rate (100 Epoch):
-------------------------------------------------------
Learning Rate: 0.0001  | Loss Train Akhir: 1.154210 | Loss Test: 0.984215
Learning Rate: 0.001   | Loss Train Akhir: 0.081542 | Loss Test: 0.041285
Learning Rate: 0.01    | Loss Train Akhir: 0.000854 | Loss Test: 0.000941
Learning Rate: 0.1     | Loss Train Akhir: 0.000042 | Loss Test: 0.000089
-------------------------------------------------------


### Tugas 4: Pembahasan Hasil Eksperimen Learning Rate
- **Learning Rate Sangat Kecil (0.0001):** Model berjalan sangat lambat, setelah 100 epoch loss masih sangat besar (1.1542) karena langkah penyesuaian bobot (*step size*) terlalu kecil untuk mendekati titik optimal.
- **Learning Rate Bawaan (0.001):** Konvergensi mulai terjadi dengan baik, menghasilkan loss 0.0412.
- **Learning Rate 0.01 & 0.1:** Pada dataset sederhana ini, learning rate 0.01 dan 0.1 membawa model menuju konvergensi yang sangat cepat dan presisi dengan loss mencapai $4.2 \times 10^{-5}$. Namun pada dataset yang kompleks, learning rate yang terlalu besar (seperti >0.1) berisiko menyebabkan osilasi (*overshooting*).


---
## Bagian 2: MLP Regresi Menggunakan Boston Housing Dataset
Sesuai modul praktikum:
1. Load dataset Boston.
2. Preprocess dengan StandardScaler dan Split data (80% train, 20% validation).
3. Bangun model Sequential: Dense(64, relu) -> Dense(32, relu) -> Dense(1).
4. Kompilasi optimizer Adam(1e-3), loss MSE, metrics MAE. Latih selama 200 epochs dengan batch_size 32.
5. Visualisasi grafik pergerakan MSE dan MAE, serta cetak metrik evaluasi RMSE data validasi.


In [3]:
# MLP regresi Boston Housing (Keras)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error

# 1. Load Dataset Boston Housing
# Kompatibilitas cerdas: load_boston() atau fetch dari CMU StatLib jika versi sklearn >= 1.2
try:
    from sklearn.datasets import load_boston
    data = load_boston()
    X = data.data
    y = data.target
    print("Dataset Boston dimuat via sklearn.datasets.load_boston")
except ImportError:
    # Alternatif resmi pembacaan data Boston Housing asli
    data_url = "http://lib.stat.cmu.edu/datasets/boston"
    raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)
    X = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
    y = raw_df.values[1::2, 2]
    print("Dataset Boston dimuat via CMU StatLib (Kompatibilitas scikit-learn modern)")

print(f"Ukuran data X: {X.shape}, Ukuran target y: {y.shape}")

# 2. Preprocess & Split
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
X_train, X_val, y_train, y_val = train_test_split(Xs, y, test_size=0.2, random_state=42)
print(f"Data latih: {X_train.shape[0]}, Data validasi: {X_val.shape[0]}")

# 3. Build model
model_boston = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],), name='dense_1'),
    Dense(32, activation='relu', name='dense_2'),
    Dense(1, name='output_price')
])

model_boston.compile(optimizer=Adam(1e-3), loss='mse', metrics=['mae'])
model_boston.summary()


Dataset Boston dimuat via sklearn.datasets.load_boston
Ukuran data X: (506, 13), Ukuran target y: (506,)
Data latih: 404, Data validasi: 102
Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_1 (Dense)             (None, 64)                896       
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 output_price (Dense)        (None, 1)                 33        
                                                                 
Total params: 3009 (11.75 KB)
Trainable params: 3009 (11.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [4]:
# 4. Train Model Boston Housing
np.random.seed(42)
tf.random.set_seed(42)

h = model_boston.fit(
    X_train, 
    y_train, 
    validation_data=(X_val, y_val), 
    epochs=200, 
    batch_size=32, 
    verbose=0
)
print("Pelatihan 200 epoch selesai.")


Pelatihan 200 epoch selesai.


In [5]:
# 5. Visualisasi Plot Kurva Training vs Validasi
plt.figure(figsize=(12, 4))

# Plot MSE Loss
plt.subplot(1, 2, 1)
plt.plot(h.history['loss'], label='train_loss (MSE)', color='blue')
plt.plot(h.history['val_loss'], label='val_loss (MSE)', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.title('Kurva Pembelajaran: Mean Squared Error (MSE)')
plt.grid(True, linestyle='--', alpha=0.6)

# Plot MAE
plt.subplot(1, 2, 2)
plt.plot(h.history['mae'], label='train_mae', color='green')
plt.plot(h.history['val_mae'], label='val_mae', color='red')
plt.xlabel('Epoch')
plt.ylabel('MAE ($ ribuan)')
plt.legend()
plt.title('Kurva Pembelajaran: Mean Absolute Error (MAE)')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# Hitung RMSE pada data validasi
pred = model_boston.predict(X_val)
rmse_val = np.sqrt(mean_squared_error(y_val, pred))
mae_val = h.history['val_mae'][-1]
mse_val = h.history['val_loss'][-1]

print(f"\n=== HASIL EVALUASI MODEL REGRESI BOSTON HOUSING ===")
print(f"Validation MSE  : {mse_val:.4f}")
print(f"Validation MAE  : {mae_val:.4f}")
print(f"Validation RMSE : {rmse_val:.4f}")


4/4 [==============================] - 0s 1ms/step

=== HASIL EVALUASI MODEL REGRESI BOSTON HOUSING ===
Validation MSE  : 11.2458
Validation MAE  : 2.1482
Validation RMSE : 3.3535


### Kesimpulan Praktikum 3:
1. **Regresi Menggunakan JST:** Pada arsitektur regresi dengan Keras, fungsi aktivasi linier pada output layer memungkinkan prediksi nilai kontinu secara mulus. Standarisasi (*StandardScaler*) sangat krusial agar fitur-fitur berdimensi beda (seperti luas, pajak, rasio kriminal) tidak mendominasi gradien bobot.
2. **Evaluasi Error Boston Housing:** Model MLP yang dilatih selama 200 epoch berhasil memprediksi harga rumah di Boston dengan rata-rata kesalahan mutlak (**MAE**) sekitar **2.14** (setara dengan $\approx \$2,140) dan **RMSE** sebesar **3.35**, yang menunjukkan akurasi regresi yang sangat kompetitif dan minim overfitting.
